# Import Img

In [10]:
import cv2
import numpy as np

img = cv2.imread("image.jpeg", cv2.IMREAD_GRAYSCALE)

In [ ]:
cv2.imwrite("blackwhite.jpg", img) 

True

In [ ]:
img.dtype

dtype('uint8')

# Canny

### Step 1: Gaussian Smoothing

In [24]:
blur = cv2.GaussianBlur(img, (5, 5), 0)
cv2.imwrite("canny/step1_gaussian.jpg", blur) 

True

### Step 2: Gradient Computation (Sobel)

In [ ]:
# Convert to float for safe math
blur = blur.astype(np.float32)

Gx_kernel = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

Gy_kernel = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

H, W = blur.shape
gx = np.zeros((H, W), dtype=np.float32)
gy = np.zeros((H, W), dtype=np.float32)

# --------- convolution with loops ---------
for i in range(1, H-1):
    for j in range(1, W-1):
        region = blur[i-1:i+2, j-1:j+2]

        gx[i, j] = np.sum(region * Gx_kernel)
        gy[i, j] = np.sum(region * Gy_kernel)

# Gradient magnitude & direction
mag = np.sqrt(gx**2 + gy**2)
angle = np.arctan2(gy, gx)

# Normalize for visualization
mag_vis = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

cv2.imwrite("canny/step2_sobel_gx.png", gx)
cv2.imwrite("canny/step2_sobel_gy.png", gy)
cv2.imwrite("canny/step2_sobel_magnitude.png", mag_vis)

print("Done — Sobel computed and saved.")


[ WARN:0@3281.376] global loadsave.cpp:1063 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.


Done — manual Sobel computed and saved.


### Step 3: Non Maximum Suppression

In [26]:
def non_max_suppression(mag, angle):
    H, W = mag.shape
    Z = np.zeros((H,W), dtype=np.float32)

    angle_deg = np.rad2deg(angle)
    angle_deg[angle_deg < 0] += 180

    for i in range(1, H-1):
        for j in range(1, W-1):
            q = 255
            r = 255

            # 0 degrees
            if (0 <= angle_deg[i,j] < 22.5) or (157.5 <= angle_deg[i,j] <= 180):
                q = mag[i, j+1]
                r = mag[i, j-1]

            # 45 degrees
            elif (22.5 <= angle_deg[i,j] < 67.5):
                q = mag[i+1, j-1]
                r = mag[i-1, j+1]

            # 90 degrees
            elif (67.5 <= angle_deg[i,j] < 112.5):
                q = mag[i+1, j]
                r = mag[i-1, j]

            # 135 degrees
            elif (112.5 <= angle_deg[i,j] < 157.5):
                q = mag[i-1, j-1]
                r = mag[i+1, j+1]

            if (mag[i,j] >= q) and (mag[i,j] >= r):
                Z[i,j] = mag[i,j]
            else:
                Z[i,j] = 0

    return Z

In [ ]:
nms = non_max_suppression(mag, angle)
nms_norm = cv2.normalize(nms, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
cv2.imwrite("canny/step3_nms.png", nms_norm)

True

### Double Threshold

In [28]:
low = 0.1 * nms.max()
high = 0.3 * nms.max()

weak = 50
strong = 255

res = np.zeros_like(nms, dtype=np.uint8)

strong_i, strong_j = np.where(nms >= high)
weak_i, weak_j = np.where((nms <= high) & (nms >= low))

res[strong_i, strong_j] = strong
res[weak_i, weak_j] = weak

cv2.imwrite("canny/step4_double_threshold.png", res)

True

### Edge Tracking by Hysteresis

In [ ]:
def hysteresis(img):
    weak = 50
    strong = 255
    H, W = img.shape
    for i in range(1, H-1):
        for j in range(1, W-1):
            if img[i,j] == weak:
                if 255 in img[i-1:i+2, j-1:j+2]:
                    img[i,j] = strong
                else:
                    # its not an edge
                    img[i,j] = 0
    return img

final_edges = hysteresis(res)
cv2.imwrite("canny/step5_hysteresis.png", final_edges)

True